In [ ]:
# from idarkvec.utils import *
from idarkvec.models.word2vec import Word2Vec
from idarkvec.preprocessing import CorpusExtractor_pl

In [ ]:
from gensim.models import Word2Vec as W2V
from multiprocessing import cpu_count
import os
from copy import deepcopy
import numpy as np
import json
import matplotlib.pyplot as plt
import polars as pl
import pandas as pd

In [ ]:
TRACES = '<TRACE_LOG_PATH>'
darknet = '<DARKNET_NAME>'
START = '20250801'

In [ ]:
from datetime import datetime, timedelta
def get_day(i, startfrom=START):
    start = datetime.strptime(startfrom, '%Y%m%d')
    DATE = start + timedelta(days=i)
    day = DATE.strftime('%Y%m%d')
    return day

In [ ]:
def get_day_vocab(corpus):
    raw_words = np.hstack(corpus)
    effective_words = np.unique(raw_words)
    return effective_words

In [ ]:
class WordCache:
    def __init__(self, ttl_days=31):
        self.ttl_days = ttl_days
        self.cache = {}  # word -> last_seen date
        
    def parse_yyyymmdd(self, day):
        dt = datetime.strptime(day, '%Y%m%d')
        return dt
    
    def update(self, words, day):
        day_dt = self.parse_yyyymmdd(day)
        for w in words:
            self.cache[w] = day_dt
            
        cutoff = day_dt - timedelta(days=self.ttl_days)
        to_delete = [w for w, d in self.cache.items() if d <= cutoff]
        print(f'[DELETE] Remove {len(to_delete)} IP from vocabulary...')
        for w in to_delete:
            del self.cache[w]

    def get_words(self):
        return list(self.cache.keys())

In [ ]:
def model_removal(pre_model, new_vocab):
    new_model = W2V(sentences=None, vector_size=pre_model.vector_size, 
                         window=pre_model.window, epochs=1, 
                         workers=cpu_count(), min_count=0, sg=1, negative=5, 
                         sample=0, seed=pre_model.seed)
    new_model.build_vocab([list(new_vocab)])
    new_model.wv.__setitem__(list(new_vocab), pre_model.wv[list(new_vocab)])
    for IP in new_vocab:
        idx_new = new_model.wv.get_index(IP)
        idx_pre = pre_model.wv.get_index(IP)
        new_model.syn1neg[idx_new] = pre_model.syn1neg[idx_pre]
    return new_model

### i-DarkVec training (with removing IPs not updated for long time to improve scalability)

In [ ]:
IP_cache = WordCache()
for day in range(31):
    current_day = get_day(day)
    print(f'---------------------------[{current_day}]')
    # extract corpus
    corpus = CorpusExtractor_pl(trace_path=f'{TRACES}/{darknet}', day=current_day, 
                             min_freq=9).from_darknet(top_ports=2500, verbose=True)
    vocab_day = get_day_vocab(corpus)
    # update IP cache, remove those did not update for 30 days
    IP_cache.update(vocab_day, current_day)
    
    if current_day == START:
        mname = f'{darknet}_{START}'
        word2vec = Word2Vec(c=5, e=200, destination=mname)
        word2vec.train(corpus, save=False)
    elif day<30:
        word2vec.update(corpus, save=False)
    else:
        word2vec.update(corpus, save=False)
        # remove "old ip" from model, keep only "new ip" in the vocab
        new_vocab = IP_cache.get_words()
        word2vec.model = model_removal(word2vec.model,new_vocab)
        
        Emb = word2vec.get_embeddings_pl()
        # Emb.write_csv(f'embeddings/{darknet}/{darknet}_{current_day}.csv')
        # Emb_lastday = Emb.filter(pl.col("ip").is_in(vocab_day))
        # Emb_lastday.write_csv(f'embeddings/{darknet}/only_last_day/{darknet}_{current_day}.csv')

        Emb.write_csv(f'embedding_{darknet}_{current_day}.csv')
        Emb_lastday = Emb.filter(pl.col("ip").is_in(vocab_day))
        Emb_lastday.write_csv(f'embedding_{darknet}_{current_day}_last_day.csv')